# VTT Meeting Transcript → Embedding-Ready Chunks Pipeline

**Objective:** Convert raw WebVTT meeting transcript files stored in a UC volume into structured, deterministic chunks suitable for embedding and RAG retrieval.

## Architecture (Medallion)

| Layer | Table | Purpose |
|-------|-------|--------|
| **Bronze** | `dev.callagent.bronze.vtt_raw` | Raw VTT file content, one row per file |
| **Silver** | `dev.callagent.silver.utterances` | Parsed utterances: speaker, timestamps, text, PII-redacted |
| **Gold** | `dev.callagent.gold.transcript_chunks` | Deterministic chunks ready for embedding |

## Deterministic Chunking Strategy

1. Parse VTT into ordered utterances (speaker, start/end time, text)
2. Accumulate consecutive utterances into a chunk until `target_chars` is reached
3. Start a new chunk, carrying the **last utterance** of the previous chunk as overlap context
4. Each chunk gets a deterministic `chunk_id` = `SHA256(call_id || chunk_index)`
5. Chunk text is formatted with speaker labels

In [0]:
import re
import hashlib
import json
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

# ── Catalog & Schema ──────────────────────────────────────────────
CATALOG = "dev"
BRONZE_SCHEMA = "callagent_bronze"
SILVER_SCHEMA = "callagent_silver"
GOLD_SCHEMA   = "callagent_gold"

# ── Source Volume ─────────────────────────────────────────────────
VOLUME_ROOT = f"/Volumes/{CATALOG}/callagent/raw/teams_transcripts"

# ── Table Names ──────────────────────────────────────────────────
BRONZE_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.vtt_raw"
SILVER_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.utterances"
GOLD_TABLE   = f"{CATALOG}.{GOLD_SCHEMA}.transcript_chunks"

# ── Chunking Parameters ──────────────────────────────────────────
TARGET_CHARS   = 2000   # target chunk size in characters (~500 tokens)
MIN_CHUNK_CHARS = 200   # don't emit a chunk smaller than this
OVERLAP_UTTS    = 1     # number of utterances to carry as overlap

# ── Create schemas if needed ─────────────────────────────────────
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}")

print("✅ Configuration loaded")
print(f"   Source volume : {VOLUME_ROOT}")
print(f"   Bronze table  : {BRONZE_TABLE}")
print(f"   Silver table  : {SILVER_TABLE}")
print(f"   Gold table    : {GOLD_TABLE}")
print(f"   Chunk target  : {TARGET_CHARS} chars (~{TARGET_CHARS // 4} tokens)")
